# Feature Selection — Fase 1: Evaluación directa multi-ventana

In [ ]:
"""Feature Selection — Fase 1: Evaluación directa multi-ventana.

Evalúa los 5 conjuntos candidatos de características (A-E) sobre los
140 motores de entrenamiento para 4 tamaños de ventana {15, 20, 25, 30}.
Registra la varianza explicada acumulada del PCA global (n_components=15)
para cada combinación conjunto × window_size.

Decisión automática al final:
    - Si ranking estable Y diferencia ≥ 2% → selección directa
    - Si no → recomendar Fase 2 (bootstrap pareado)

Run desde el proyecto root:
    python feature_selection_phase1.py
"""

import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA

from src.dataset_manager import DatasetManager
from src.pipeline.windowing import build_windows, flatten_windows
from src.pipeline.feature_extraction import (
    extract_window_features,
    FEATURE_SETS,
)

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------
WINDOW_SIZES   = [15, 20, 25, 30]
N_COMPONENTS   = 15
CLIPPING       = 125
THRESHOLD_PCT  = 2.0   # % mínimo de diferencia para selección directa

print("=" * 65)
print("FEATURE SELECTION — Fase 1: Evaluación Directa Multi-Ventana")
print("=" * 65)

# ---------------------------------------------------------------------------
# Cargar los 140 motores de entrenamiento
# ---------------------------------------------------------------------------
print("\nCargando motores de entrenamiento...", end=' ', flush=True)
t0 = time.perf_counter()

m_train, _ = DatasetManager.split_dataset()

dfs = []
for idx in m_train:
    df = pd.read_csv(f'data/clean/data_motor_{idx}.csv')
    df.insert(0, 'unit_number', idx)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
X_df = df_all.drop(columns=['RUL'])

print(f"{time.perf_counter()-t0:.1f}s")
print(f"  Motores: {X_df['unit_number'].nunique()}, "
      f"Filas: {len(X_df)}")

# ---------------------------------------------------------------------------
# Evaluación: conjuntos × window_sizes
# ---------------------------------------------------------------------------
print(f"\n{'─'*65}")
print(f"EVALUACIÓN: {len(FEATURE_SETS)} conjuntos × {len(WINDOW_SIZES)} window_sizes")
print(f"{'─'*65}")

# Matriz de resultados: cumvar[conjunto][window_size]
results: dict[str, dict[int, float]] = {
    name: {} for name in FEATURE_SETS
}

# Timing por conjunto
timing: dict[str, float] = {name: 0.0 for name in FEATURE_SETS}

for ws in WINDOW_SIZES:
    print(f"\n  window_size = {ws}")

    # Nodo 2 — build_windows (una sola vez por window_size)
    t0 = time.perf_counter()
    motor_windows = build_windows(X_df, window_size=ws, clipping_threshold=CLIPPING)
    t_wind = time.perf_counter() - t0
    n_windows = sum(d['X_windows'].shape[0] for d in motor_windows.values())
    print(f"    Windowing: {t_wind:.2f}s ({n_windows} ventanas)")

    for name, features in FEATURE_SETS.items():

        # Nodo 3 — extract features
        t0 = time.perf_counter()
        motor_features = extract_window_features(motor_windows, features=features)
        t_feat = time.perf_counter() - t0

        # Concatenar globalmente
        X_all = np.concatenate(
            [d['X_windows'] for d in motor_features.values()],
            axis=0
        )

        # RobustScaler + PCA
        X_scaled = RobustScaler().fit_transform(X_all)
        pca = PCA(n_components=min(N_COMPONENTS, X_scaled.shape[1]))
        pca.fit(X_scaled)
        cumvar = float(pca.explained_variance_ratio_.cumsum()[-1])

        results[name][ws] = cumvar
        timing[name] += t_feat

        print(f"    [{name}] features={len(features)}×16={len(features)*16:3d}  "
              f"feat_time={t_feat:.1f}s  cumvar={cumvar:.4f}")

# ---------------------------------------------------------------------------
# Tabla de resultados
# ---------------------------------------------------------------------------
print(f"\n{'─'*65}")
print("TABLA DE RESULTADOS — cumvar (varianza explicada acumulada, n_components=15)")
print(f"{'─'*65}")

df_results = pd.DataFrame(results).T
df_results.columns = [f'ws={ws}' for ws in WINDOW_SIZES]
df_results.index.name = 'Conjunto'

# Añadir columnas de análisis
df_results['media'] = df_results.mean(axis=1)
df_results['std']   = df_results.std(axis=1)
df_results['min']   = df_results[[f'ws={ws}' for ws in WINDOW_SIZES]].min(axis=1)
df_results['max']   = df_results[[f'ws={ws}' for ws in WINDOW_SIZES]].max(axis=1)

print(f"\n{df_results.round(4).to_string()}")

# ---------------------------------------------------------------------------
# Análisis de ranking
# ---------------------------------------------------------------------------
print(f"\n{'─'*65}")
print("ANÁLISIS DE RANKING POR WINDOW_SIZE")
print(f"{'─'*65}")

rankings = {}
for ws in WINDOW_SIZES:
    col = f'ws={ws}'
    ranking = df_results[col].sort_values(ascending=False)
    rankings[ws] = list(ranking.index)
    print(f"  ws={ws}: {' > '.join(rankings[ws])}")

# ¿El top-2 del ranking es estable?
# Solo importa que el ganador y el segundo sean consistentes
# — intercambios entre posiciones 3-5 son irrelevantes para la decisión.
top2_stable = all(
    rankings[ws][:2] == rankings[WINDOW_SIZES[0]][:2]
    for ws in WINDOW_SIZES[1:]
)
ranking_stable = top2_stable
winner = rankings[WINDOW_SIZES[0]][0]

# ¿La diferencia entre el ganador y el segundo es ≥ umbral?
second = rankings[WINDOW_SIZES[0]][1]
diffs = []
for ws in WINDOW_SIZES:
    col = f'ws={ws}'
    diff = (df_results.loc[winner, col] - df_results.loc[second, col]) * 100
    diffs.append(diff)
min_diff = min(diffs)
clear_winner = min_diff >= THRESHOLD_PCT

# ---------------------------------------------------------------------------
# Decisión automática
# ---------------------------------------------------------------------------
print(f"\n{'─'*65}")
print("DECISIÓN AUTOMÁTICA")
print(f"{'─'*65}")
print(f"  Ganador:              Conjunto {winner}")
print(f"  Segundo:              Conjunto {second}")
print(f"  Ranking top-2 estable:  {'✅ SÍ' if ranking_stable else '❌ NO'}")
print(f"  Diferencia mínima:    {min_diff:.2f}%  (umbral: {THRESHOLD_PCT}%)")
print(f"  Diferencia clara:     {'✅ SÍ' if clear_winner else '❌ NO'}")

if ranking_stable and clear_winner:
    print(f"\n  ✅ SELECCIÓN DIRECTA: Conjunto {winner}")
    print(f"     Fase 2 (bootstrap) NO requerida.")
    best_ws = df_results.loc[winner, [f'ws={ws}' for ws in WINDOW_SIZES]].idxmax()
    best_ws_val = int(best_ws.split('=')[1])
    print(f"     Mejor window_size para {winner}: {best_ws_val}")
else:
    print(f"\n  ⚠️  FASE 2 REQUERIDA (bootstrap pareado)")
    print(f"     Conjuntos finalistas: {winner}, {second}")
    if not ranking_stable:
        print(f"     Razón: ranking inestable entre window_sizes")
    if not clear_winner:
        print(f"     Razón: diferencia mínima {min_diff:.2f}% < {THRESHOLD_PCT}%")

# ---------------------------------------------------------------------------
# Timing summary
# ---------------------------------------------------------------------------
print(f"\n{'─'*65}")
print("TIMING DE EXTRACCIÓN (suma sobre todos los window_sizes)")
print(f"{'─'*65}")
for name, t in timing.items():
    n_feat = len(FEATURE_SETS[name]) * 16
    print(f"  Conjunto {name}: {t:.1f}s  ({n_feat} características)")

# ---------------------------------------------------------------------------
# Gráfico
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Feature Selection — Fase 1: cumvar vs window_size', fontsize=13)

colors = {'A': 'gray', 'B': 'steelblue', 'C': 'darkorange',
          'D': 'seagreen', 'E': 'crimson'}

# Gráfico 1 — cumvar vs window_size por conjunto
ax = axes[0]
for name in FEATURE_SETS:
    vals = [results[name][ws] for ws in WINDOW_SIZES]
    ax.plot(WINDOW_SIZES, vals, marker='o', linewidth=2,
            color=colors[name], label=f'Conjunto {name}')
ax.axhline(y=0.5, color='black', linestyle=':', alpha=0.4,
           label='50% (referencia Alomari)')
ax.set_xlabel('window_size (ciclos)')
ax.set_ylabel('cumvar (varianza explicada acumulada)')
ax.set_title('cumvar por conjunto y window_size')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_xticks(WINDOW_SIZES)

# Gráfico 2 — barras de media ± std por conjunto
ax = axes[1]
names  = list(FEATURE_SETS.keys())
medias = [df_results.loc[n, 'media'] for n in names]
stds   = [df_results.loc[n, 'std']   for n in names]
bars = ax.bar(names, medias, yerr=stds, capsize=5,
              color=[colors[n] for n in names], alpha=0.8)
ax.set_xlabel('Conjunto de características')
ax.set_ylabel('cumvar media ± std')
ax.set_title('Resumen: media y variabilidad por conjunto')
ax.grid(alpha=0.3, axis='y')
for bar, val in zip(bars, medias):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('feature_selection_phase1.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nGráfico guardado: feature_selection_phase1.png")